In [ ]:
# ======================================================================
# CAPÍTULO 4 – IMPORTS, ESTILO E PARÂMETROS
# ======================================================================

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage
from pathlib import Path
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

# Estilo dos gráficos (visual mais profissional para tese)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (16, 10)
plt.rcParams["font.size"] = 12
plt.rcParams["font.family"] = "serif"
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["lines.linewidth"] = 2.5
plt.rcParams["lines.markersize"] = 8

# Paleta de cores padronizada (MRT vs T2F, fases e sequências)
CORES = {
    "MRT":   "#E74C3C",
    "T2F":   "#3498DB",
    "Fase_A": "#00CED1",
    "Fase_B": "#FF6347",
    "Fase_C": "#32CD32",
    "I0":    "#4169E1",
    "I1":    "#228B22",
    "I2":    "#8B008B",
}


In [ ]:
# ======================================================================
# SEÇÃO 1 – CARREGAMENTO DOS ARQUIVOS HDF5
# ======================================================================

# Caminho da pasta com os .mat (ajuste aqui quando trocar de cenário)
pasta_entrada = Path("C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_13/Processados_HDF5/")

saida_fig = Path("Figuras_Cap4_Plotly")
saida_fig.mkdir(exist_ok=True)

# Carrega lista de arquivos
arquivos = sorted(pasta_entrada.glob("*.mat"))  # ou "*.mat" / "*_py.mat"

# Lista de barras que serão analisadas
barras = ["800", "818", "820", "822", "T2F", "T2F1"]

# dicionário mestre: dados_por_arquivo["nome.mat"]["I_822"] -> array N x 3
dados_por_arquivo = {}
proc_por_arquivo_barra = {}


In [ ]:
# ======================================================================
# SEÇÃO 2 – LEITURA DOS ARQUIVOS E MONTAGEM DO DICIONÁRIO
# ======================================================================

for arq in arquivos:
    with h5py.File(arq, "r") as f:
        # Conferência básica: precisa ter vetor de tempo t
        if "t" not in f.keys():
            continue

        nome = arq.name
        dados_por_arquivo[nome] = {}

        # Vetor de tempo
        t = np.array(f["t"]).flatten()
        # Posição da falta, se existir nos arquivos
        if "m1" in f.keys():
            m1_arr = np.array(f["m1"]).flatten()
            m1 = float(m1_arr[0])
        else:
            m1 = None

        dados_por_arquivo[nome]["t"] = t
        dados_por_arquivo[nome]["m1"] = m1

        # Loop sobre barras e leitura de correntes/tensões
        for barra in barras:
            nome_i = f"I_{barra}_raw"
            nome_v = f"V_{barra}_raw"

            # Corrente da barra (se existir no .mat)
            if nome_i in f.keys():
                I = np.array(f[nome_i])
                # Garante forma N x 3 (N amostras, 3 fases)
                if I.ndim == 1:
                    I = np.column_stack([I] * 3)
                elif I.shape[0] == 3 and I.shape[1] > 3:
                    I = I.T
                dados_por_arquivo[nome][f"I_{barra}"] = I

            # Tensão da barra (se existir no .mat)
            if nome_v in f.keys():
                V = np.array(f[nome_v])
                if V.ndim == 1:
                    V = np.column_stack([V] * 3)
                elif V.shape[0] == 3 and V.shape[1] > 3:
                    V = V.T
                dados_por_arquivo[nome][f"V_{barra}"] = V

print("Arquivos carregados:", len(dados_por_arquivo))
list(dados_por_arquivo.keys())[:5]


In [ ]:
# ======================================================================
# SEÇÃO 3 – PROCESSADOR DE UMA BARRA (RMS, CLARKE, SEQUÊNCIAS)
# ======================================================================

class BarraProcessor:
    """
    Processa tensões e correntes de uma barra específica:
    - Ajusta formato dos sinais (N x 3).
    - Calcula RMS deslizante (janela de 1 ciclo).
    - Calcula transformação de Clarke das correntes.
    - Calcula componentes de sequência (Fortescue): I0, I1, I2.
    """

    def __init__(self, t, v_raw, i_raw, freq=60):
        # Garante arrays 1D e 2D com tipo float
        t = np.asarray(t, dtype=float).flatten()
        v_raw = np.asarray(v_raw, dtype=float)
        i_raw = np.asarray(i_raw, dtype=float)

        self.t = t
        self.freq = float(freq)

        # Ajusta forma dos vetores (sempre N x 3)
        self.v_raw = self._fix_shape(v_raw, len(t)) / np.sqrt(3.0)
        self.i_raw = self._fix_shape(i_raw, len(t))

        # Corta todos com o mesmo comprimento L
        L = min(len(self.t), len(self.v_raw), len(self.i_raw))
        if L < 10:
            raise ValueError("Dados insuficientes.")
        self.t = self.t[:L]
        self.v_raw = self.v_raw[:L]
        self.i_raw = self.i_raw[:L]

        # Passo de tempo e taxa de amostragem
        dt = self.t[1] - self.t[0]
        self.dt = float(dt)
        self.fs = 1.0 / self.dt

        # Número de amostras em 1 ciclo (para RMS deslizante)
        self.samples = max(1, int(self.fs / self.freq))

        # RMS deslizante das três fases
        self.v_rms = self._rms(self.v_raw)
        self.i_rms = self._rms(self.i_raw)

        # Transformação de Clarke das correntes (alpha, beta)
        self.i_clarke = self._clarke(self.i_raw)

        # Componentes de sequência (Fortescue): magnitudes I0, I1, I2
        self.I0, self.I1, self.I2 = self._seq_components(self.i_raw)
        self.i_seq = np.stack([self.I0, self.I1, self.I2], axis=1)

    def _fix_shape(self, mat, N):
        if mat.ndim == 1:
            col = mat.flatten()
            return np.stack([col, col, col], axis=1)
        if mat.ndim == 2:
            if mat.shape[0] == 3 and mat.shape[1] > 3:
                return mat.T
            if mat.shape[1] == 3:
                return mat
        return np.zeros((N, 3))

    def _rms(self, x):
        return np.sqrt(np.abs(
            scipy.ndimage.uniform_filter1d(x**2, self.samples, axis=0)
        ))

    def _clarke(self, abc):
        a, b, c = abc[:, 0], abc[:, 1], abc[:, 2]
        alpha = (2*a - b - c)/3.0
        beta  = (b - c)/np.sqrt(3.0)
        return {"alpha": alpha, "beta": beta}

    def _seq_components(self, x):
        rot = np.exp(-1j * 2*np.pi * self.freq * self.t)
        ph = np.zeros_like(x, dtype=complex)

        for k in range(3):
            ph[:, k] = scipy.ndimage.uniform_filter1d(
                x[:, k] * rot, self.samples
            ) * np.sqrt(2.0)

        a = np.exp(1j * 2*np.pi / 3.0)

        I0 = (ph[:, 0] +      ph[:, 1] +      ph[:, 2]) / 3.0
        I1 = (ph[:, 0] +  a * ph[:, 1] + a**2*ph[:, 2]) / 3.0
        I2 = (ph[:, 0] + a**2*ph[:, 1] +  a *ph[:, 2]) / 3.0

        return np.abs(I0), np.abs(I1), np.abs(I2)


In [ ]:
# ======================================================================
# SEÇÃO 4 – PROCESSAR TODAS AS COMBINAÇÕES (ARQUIVO, BARRA)
# ======================================================================

proc_por_arquivo_barra = {}

for nome_arq, dados in dados_por_arquivo.items():
    t = dados["t"]
    for barra in barras:
        chave_I = f"I_{barra}"
        chave_V = f"V_{barra}"

        if chave_I in dados and chave_V in dados:
            v = dados[chave_V]
            i = dados[chave_I]

            L = min(len(t), len(v), len(i))
            if L < 10:
                continue

            proc = BarraProcessor(t, v, i)
            proc_por_arquivo_barra[(nome_arq, barra)] = proc

print("Total de combinações arquivo-barra processadas:",
      len(proc_por_arquivo_barra))

nome_exemplo = "Qualificacao_SR__R_822_-_Falta_ABC_py.mat"
barras_disponiveis = [k[1] for k in proc_por_arquivo_barra.keys()
                      if k[0] == nome_exemplo]
print("Barras processadas em", nome_exemplo, ":", barras_disponiveis)


In [ ]:
# ======================================================================
# SEÇÃO 5 – CRIAR DATAFRAME COM MÉTRICAS RESUMIDAS (df_resumo)
# ======================================================================

linhas = []

for (nome_arq, barra), p in proc_por_arquivo_barra.items():
    I_pico = np.max(p.i_rms, axis=0)
    I_pico_max = float(np.max(p.i_rms))

    m1 = dados_por_arquivo[nome_arq]["m1"]

    if nome_arq.startswith("MRT"):
        tipo_sistema = "MRT"
    else:
        tipo_sistema = "T2F"

    if "Falta_ABC" in nome_arq:
        tipo_falta = "ABC"
    elif "Falta_AB" in nome_arq:
        tipo_falta = "AB"
    elif "Falta_AC" in nome_arq:
        tipo_falta = "AC"
    elif "Falta_BC" in nome_arq:
        tipo_falta = "BC"
    elif "Falta_A" in nome_arq:
        tipo_falta = "A-G"
    elif "Sem_Falta" in nome_arq or "Normal" in nome_arq:
        tipo_falta = "Normal"
    else:
        tipo_falta = "Desconhecido"

    linhas.append({
        "arquivo": nome_arq,
        "barra": barra,
        "m1": m1,
        "tipo_sistema": tipo_sistema,
        "tipo_falta": tipo_falta,
        "I_pico_A": I_pico[0],
        "I_pico_B": I_pico[1],
        "I_pico_C": I_pico[2],
        "I_pico_max": I_pico_max,
        "I0_max": float(np.max(p.I0)),
        "I1_max": float(np.max(p.I1)),
        "I2_max": float(np.max(p.I2)),
    })

df_resumo = pd.DataFrame(linhas)
df_resumo.head(10)


In [ ]:
# ======================================================================
# SEÇÃO 6 – GRÁFICOS DETALHADOS (UM CASO, UMA BARRA)
# ======================================================================

nome = "Qualificacao__R_822_-_Falta_ABC_py.mat"
barra = "822"

p = proc_por_arquivo_barra[(nome, barra)]

# 6.1 – Tensão RMS na barra
plt.figure(figsize=(10, 4))
plt.plot(p.t, p.v_rms[:, 0], label="Va RMS", color=CORES["Fase_A"])
plt.plot(p.t, p.v_rms[:, 1], label="Vb RMS", color=CORES["Fase_B"])
plt.plot(p.t, p.v_rms[:, 2], label="Vc RMS", color=CORES["Fase_C"])
plt.xlabel("Tempo (s)")
plt.ylabel("Tensão RMS (V)")
plt.title(f"Tensões RMS – Barra {barra} – {nome}")
plt.grid(True); plt.legend()
plt.tight_layout()
plt.show()

# 6.2 – Corrente RMS na barra
plt.figure(figsize=(10, 4))
plt.plot(p.t, p.i_rms[:, 0], label="Ia RMS", color=CORES["Fase_A"])
plt.plot(p.t, p.i_rms[:, 1], label="Ib RMS", color=CORES["Fase_B"])
plt.plot(p.t, p.i_rms[:, 2], label="Ic RMS", color=CORES["Fase_C"])
plt.xlabel("Tempo (s)")
plt.ylabel("Corrente RMS (A)")
plt.title(f"Correntes RMS – Barra {barra} – {nome}")
plt.grid(True); plt.legend()
plt.tight_layout()
plt.show()

# 6.3 – Transformação de Clarke (XY)
plt.figure(figsize=(5, 5))
plt.plot(p.i_clarke["alpha"], p.i_clarke["beta"], color=CORES["MRT"])
plt.xlabel("Alpha")
plt.ylabel("Beta")
plt.title(f"Clarke XY – Barra {barra} – {nome}")
plt.grid(True); plt.axis("equal")
plt.tight_layout()
plt.show()

# 6.4 – Componentes de sequência (I0, I1, I2) no tempo
plt.figure(figsize=(10, 4))
plt.plot(p.t, p.I0, label="I0 (seq. zero)", color=CORES["I0"])
plt.plot(p.t, p.I1, label="I1 (seq. positiva)", color=CORES["I1"])
plt.plot(p.t, p.I2, label="I2 (seq. negativa)", color=CORES["I2"])
plt.xlabel("Tempo (s)")
plt.ylabel("Corrente (A)")
plt.title(f"Componentes de sequência – Barra {barra} – {nome}")
plt.grid(True); plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================================
# SEÇÃO 7 – COMPARAÇÃO GLOBAL MRT × T2F EM FUNÇÃO DE m1 (TODAS AS BARRAS)
# ======================================================================

# 7.1 – Ipico_max × m1 para cada barra (MRT A-G vs T2F ABC)

for barra in barras:
    df_mrt = df_resumo[
        (df_resumo["tipo_sistema"] == "MRT") &
        (df_resumo["tipo_falta"] == "A-G") &
        (df_resumo["barra"] == barra)
    ]
    df_t2f = df_resumo[
        (df_resumo["tipo_sistema"] == "T2F") &
        (df_resumo["tipo_falta"] == "ABC") &
        (df_resumo["barra"] == barra)
    ]

    plt.figure(figsize=(7, 4))
    if not df_mrt.empty:
        plt.plot(df_mrt["m1"] * 100, df_mrt["I_pico_max"],
                 "o-", label="MRT A-G", color=CORES["MRT"])
    if not df_t2f.empty:
        plt.plot(df_t2f["m1"] * 100, df_t2f["I_pico_max"],
                 "s-", label="T2F ABC", color=CORES["T2F"])
    plt.xlabel("Posição da falta m1 (%)")
    plt.ylabel(f"I pico RMS – barra {barra} (A)")
    plt.title(f"Corrente de curto em função de m1 – Barra {barra}")
    plt.grid(True); plt.legend()
    plt.tight_layout()
    plt.show()

# 7.2 – I0, I1, I2 × m1 para cada barra (MRT vs T2F)

for barra in barras:
    df_mrt = df_resumo[
        (df_resumo["tipo_sistema"] == "MRT") &
        (df_resumo["tipo_falta"] == "A-G") &
        (df_resumo["barra"] == barra)
    ]
    df_t2f = df_resumo[
        (df_resumo["tipo_sistema"] == "T2F") &
        (df_resumo["tipo_falta"] == "ABC") &
        (df_resumo["barra"] == barra)
    ]

    plt.figure(figsize=(7, 4))
    if not df_mrt.empty:
        plt.plot(df_mrt["m1"] * 100, df_mrt["I0_max"], "o-", label="MRT I0", color=CORES["I0"])
        plt.plot(df_mrt["m1"] * 100, df_mrt["I1_max"], "s-", label="MRT I1", color=CORES["I1"])
        plt.plot(df_mrt["m1"] * 100, df_mrt["I2_max"], "^-", label="MRT I2", color=CORES["I2"])
    if not df_t2f.empty:
        plt.plot(df_t2f["m1"] * 100, df_t2f["I0_max"], "o--", label="T2F I0", color=CORES["I0"])
        plt.plot(df_t2f["m1"] * 100, df_t2f["I1_max"], "s--", label="T2F I1", color=CORES["I1"])
        plt.plot(df_t2f["m1"] * 100, df_t2f["I2_max"], "^--", label="T2F I2", color=CORES["I2"])
    plt.xlabel("Posição da falta m1 (%)")
    plt.ylabel(f"Componentes de sequência – barra {barra} (A)")
    plt.title(f"I0, I1, I2 em função de m1 – Barra {barra}")
    plt.grid(True); plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ======================================================================
# PLOTLY 1 – I_pico_max x m1 (todos arquivos, todas as barras) + SALVAR
# ======================================================================

for barra in barras:
    df_barra = df_resumo[
        (df_resumo["barra"] == barra) &
        (df_resumo["tipo_falta"].isin(["A-G", "ABC"]))
    ].copy()
    if df_barra.empty:
        continue

    df_barra["m1_pct"] = df_barra["m1"] * 100.0

    fig = px.scatter(
        df_barra,
        x="m1_pct",
        y="I_pico_max",
        color="tipo_sistema",          # MRT x T2F
        symbol="tipo_falta",           # A-G, ABC
        hover_data=["arquivo"],
        title=f"I pico RMS x m1 – Barra {barra} (MRT A-G x T2F ABC)",
        labels={
            "m1_pct": "Posição da falta m1 (%)",
            "I_pico_max": "I pico RMS (A)",
            "tipo_sistema": "Sistema",
            "tipo_falta": "Tipo de falta",
        },
    )

    fig.update_traces(mode="lines+markers")
    fig.update_layout(template="plotly_white")

    nome_fig = saida_fig / f"Cap4_Ipico_m1_barra_{barra}.png"
    fig.write_image(str(nome_fig), scale=3)
    print("salvo:", nome_fig)


In [ ]:
# ======================================================================
# PLOTLY 2 – I0, I1, I2 x m1 (todos arquivos, todas as barras) + SALVAR
# ======================================================================

for barra in barras:
    df_barra = df_resumo[
        (df_resumo["barra"] == barra) &
        (df_resumo["tipo_falta"].isin(["A-G", "ABC"]))
    ].copy()
    if df_barra.empty:
        continue

    df_barra["m1_pct"] = df_barra["m1"] * 100.0

    df_long = pd.melt(
        df_barra,
        id_vars=["arquivo", "barra", "m1_pct", "tipo_sistema", "tipo_falta"],
        value_vars=["I0_max", "I1_max", "I2_max"],
        var_name="Sequencia",
        value_name="Corrente",
    )
    df_long["Sequencia"] = df_long["Sequencia"].map(
        {"I0_max": "I0", "I1_max": "I1", "I2_max": "I2"}
    )

    fig = px.line(
        df_long,
        x="m1_pct",
        y="Corrente",
        color="Sequencia",          # I0, I1, I2
        line_dash="tipo_sistema",   # MRT vs T2F
        hover_data=["arquivo", "tipo_falta"],
        title=f"I0, I1, I2 x m1 – Barra {barra} (MRT A-G x T2F ABC)",
        labels={
            "m1_pct": "Posição da falta m1 (%)",
            "Corrente": "Corrente (A)",
            "Sequencia": "Componente",
            "tipo_sistema": "Sistema",
        },
    )

    fig.update_layout(template="plotly_white")

    nome_fig = saida_fig / f"Cap4_Seq_m1_barra_{barra}.png"
    fig.write_image(str(nome_fig), scale=3)
    print("salvo:", nome_fig)


In [ ]:
# ======================================================================
# PLOTLY 3 – I0, I1, I2 x tempo (todos arquivos, barra escolhida)
# ======================================================================

barra_escolhida = "822"   # altere se quiser outra barra

pares = [(nome_arq, b) for (nome_arq, b) in proc_por_arquivo_barra.keys()
         if b == barra_escolhida]

for nome_arq, b in pares:
    p = proc_por_arquivo_barra[(nome_arq, b)]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=p.t, y=p.I0,
        mode="lines",
        name="I0 (seq. zero)",
        line=dict(color=CORES["I0"])
    ))
    fig.add_trace(go.Scatter(
        x=p.t, y=p.I1,
        mode="lines",
        name="I1 (seq. positiva)",
        line=dict(color=CORES["I1"])
    ))
    fig.add_trace(go.Scatter(
        x=p.t, y=p.I2,
        mode="lines",
        name="I2 (seq. negativa)",
        line=dict(color=CORES["I2"])
    ))

    fig.update_layout(
        title=f"Componentes de sequência x tempo – Barra {barra_escolhida} – {nome_arq}",
        xaxis_title="Tempo (s)",
        yaxis_title="Corrente (A)",
        template="plotly_white",
    )
    nome_fig = saida_fig / f"Cap4_Seq_vs_tempo_barra_{barra_escolhida}_{nome_arq}.png"
    fig.write_image(str(nome_fig), scale=3)
    print("salvo:", nome_fig)


In [ ]:
# ======================================================================
# PLOTLY 4 – Clarke XY para todos os arquivos e barras
# ======================================================================

for (nome_arq, barra), p in proc_por_arquivo_barra.items():
    fig = px.scatter(
        x=p.i_clarke["alpha"],
        y=p.i_clarke["beta"],
        title=f"Clarke XY – Barra {barra} – {nome_arq}",
        labels={"x": "Alpha", "y": "Beta"},
    )
    fig.update_traces(mode="lines", line=dict(color=CORES["MRT"]))
    fig.update_layout(
        template="plotly_white",
        xaxis=dict(scaleanchor="y", scaleratio=1),
    )
    nome_fig = saida_fig / f"Clarke_X_Y_barra_{barra}_{nome_arq}.png"
    fig.write_image(str(nome_fig), scale=3)
    print("salvo:", nome_fig)


In [ ]:
# ======================================================================
# SEÇÃO 8 – ANÁLISE E CATEGORIZAÇÃO DOS ARQUIVOS .MAT
# ======================================================================

import re

def analisar_estrutura_arquivos(pasta_dados):
    """
    Analisa e categoriza arquivos .mat na pasta

    PADRÕES:
    - T2F: arquivos com 'Qualificacao'
    - MRT: arquivos com 'MRT'
    - SR = Sem regulador  -> nomes contendo '_SR_'
    - sem_terra = Sem aterramento -> nomes contendo '_sem_terra_'
    """
    pasta = Path(pasta_dados)

    categorias = {
        'Qualificacao_sem_terra': [],
        'Qualificacao_com_terra': [],
        'MRT_sem_terra': [],
        'MRT_com_terra': [],
        'Qualificacao_sem_regulador': [],  # SR = Sem Regulador
        'Qualificacao_com_regulador': [],  # Sem SR = Com Regulador
        'MRT_sem_regulador': [],           # MRT com SR
        'MRT_com_regulador': [],           # MRT sem SR
        'faltas_ABC': [],
        'faltas_AB': [],
        'faltas_BC': [],
        'faltas_AC': [],
        'faltas_A': [],
        'faltas_B': [],
        'faltas_C': [],
        'sem_falta': [],
        'barras_800': [],
        'barras_818': [],
        'barras_820': [],
        'barras_822': [],
        'barras_T2F': [],
        'varredura_posicao': [],
    }

    arquivos_mat = list(pasta.glob('**/*.mat'))

    print(f"\n📂 Encontrados {len(arquivos_mat)} arquivos .mat")
    print("=" * 80)

    for arquivo in arquivos_mat:
        nome = arquivo.name
        nome_lower = nome.lower()

        # 1. SISTEMA: T2F ou MRT
        is_t2f = 'qualificacao' in nome_lower
        is_mrt = 'mrt' in nome_lower

        if not (is_t2f or is_mrt):
            continue

        # 2. ATERRAMENTO
        is_sem_terra = '_sem_terra_' in nome_lower

        if is_t2f:
            if is_sem_terra:
                categorias['Qualificacao_sem_terra'].append(arquivo)
            else:
                categorias['Qualificacao_com_terra'].append(arquivo)

        if is_mrt:
            if is_sem_terra:
                categorias['MRT_sem_terra'].append(arquivo)
            else:
                categorias['MRT_com_terra'].append(arquivo)

        # 3. REGULADOR (SR = Sem regulador)
        has_sr = '_sr_' in nome_lower

        if is_t2f:
            if has_sr:
                categorias['Qualificacao_sem_regulador'].append(arquivo)
            else:
                categorias['Qualificacao_com_regulador'].append(arquivo)

        if is_mrt:
            if has_sr:
                categorias['MRT_sem_regulador'].append(arquivo)
            else:
                categorias['MRT_com_regulador'].append(arquivo)

        # 4. TIPO DE FALTA
        if 'sem_falta' in nome_lower:
            categorias['sem_falta'].append(arquivo)
        elif 'falta_abc' in nome_lower:
            categorias['faltas_ABC'].append(arquivo)
        elif 'falta_ab' in nome_lower:
            categorias['faltas_AB'].append(arquivo)
        elif 'falta_bc' in nome_lower:
            categorias['faltas_BC'].append(arquivo)
        elif 'falta_ac' in nome_lower or 'falta_a_c' in nome_lower:
            categorias['faltas_AC'].append(arquivo)
        elif 'falta_a' in nome_lower:
            categorias['faltas_A'].append(arquivo)
        elif 'falta_b' in nome_lower:
            categorias['faltas_B'].append(arquivo)
        elif 'falta_c' in nome_lower:
            categorias['faltas_C'].append(arquivo)

        # 5. BARRAS
        if re.search(r'[_\-]800[_\-\.]|_a_800|_r_800', nome_lower):
            categorias['barras_800'].append(arquivo)
        if re.search(r'[_\-]818[_\-\.]|_a_818|_r_818', nome_lower):
            categorias['barras_818'].append(arquivo)
        if re.search(r'[_\-]820[_\-\.]|_a_820|_r_820', nome_lower):
            categorias['barras_820'].append(arquivo)
        if re.search(r'[_\-]822[_\-\.]|_a_822|_r_822', nome_lower):
            categorias['barras_822'].append(arquivo)

        if re.search(r't2f[_\-]', nome_lower) and 'qualificacao' not in nome_lower:
            categorias['barras_T2F'].append(arquivo)

        # 6. VARREDURA DE POSIÇÃO
        if re.search(r'm10p\d{2}', nome_lower):
            categorias['varredura_posicao'].append(arquivo)

    return categorias, arquivos_mat


In [ ]:
# ======================================================================
# SEÇÃO 9 – EXIBIR RESUMO DAS CATEGORIAS
# ======================================================================

def exibir_resumo_categorias(categorias):
    """
    Exibe resumo organizado das categorias
    """
    print("\n📊 RESUMO DAS CATEGORIAS")
    print("=" * 80)

    nomes_amigaveis = {
        'Qualificacao_sem_regulador': '🔴 T2F SEM Regulador (SR)',
        'Qualificacao_com_regulador': '🟢 T2F COM Regulador',
        'Qualificacao_sem_terra': '🔵 T2F Sem Terra',
        'Qualificacao_com_terra': '🟣 T2F Com Terra',
        'MRT_sem_regulador': '🔴 MRT SEM regulador (SR)',
        'MRT_com_regulador': '🟢 MRT COM regulador',
        'MRT_sem_terra': '🔵 MRT Sem Terra',
        'MRT_com_terra': '🟣 MRT Com Terra',
        'faltas_ABC': '⚡ Faltas ABC (Trifásicas)',
        'faltas_AB': '⚡ Faltas AB (Bifásicas)',
        'faltas_BC': '⚡ Faltas BC (Bifásicas)',
        'faltas_AC': '⚡ Faltas AC (Bifásicas)',
        'faltas_A': '⚡ Faltas A (Monofásicas)',
        'faltas_B': '⚡ Faltas B (Monofásicas)',
        'faltas_C': '⚡ Faltas C (Monofásicas)',
        'sem_falta': '✅ Sem Falta (Operação Normal)',
        'barras_800': '📍 Barra 800',
        'barras_818': '📍 Barra 818',
        'barras_820': '📍 Barra 820',
        'barras_822': '📍 Barra 822',
        'barras_T2F': '📍 Barra T2F',
        'varredura_posicao': '📏 Varredura de Posição (m10p)',
    }

    for categoria, arquivos in categorias.items():
        if arquivos:
            nome_exibicao = nomes_amigaveis.get(
                categoria,
                categoria.replace('_', ' ').title()
            )
            print(f"\n🔹 {nome_exibicao}: {len(arquivos)} arquivo(s)")
            for arq in arquivos[:3]:
                print(f" • {arq.name}")
            if len(arquivos) > 3:
                print(f" ... e mais {len(arquivos) - 3} arquivo(s)")


In [ ]:
# ======================================================================
# SEÇÃO 10 – SUGESTÃO DE ARQUIVOS PARA A TESE
# ======================================================================

def sugerir_arquivos_tese(categorias):
    """
    Sugere os melhores arquivos para incluir na tese
    """
    print("\n" + "=" * 80)
    print("🎓 SUGESTÃO DE ARQUIVOS PARA TESE - CAPÍTULO 4")
    print("=" * 80)

    sugestoes = {
        'Comparação T2F vs MRT': {
            'objetivo': 'Demonstrar superioridade do T2F sobre MRT',
            'arquivos': []
        },
        'Com Terra vs Sem Terra (T2F)': {
            'objetivo': 'Mostrar impacto do aterramento no T2F',
            'arquivos': []
        },
        'Com Terra vs Sem Terra (MRT)': {
            'objetivo': 'Mostrar impacto do aterramento no MRT',
            'arquivos': []
        },
        'Diferentes Tipos de Falta': {
            'objetivo': 'Caracterizar comportamento para cada tipo',
            'arquivos': []
        },
        'Comparação de Barras': {
            'objetivo': 'Mostrar variação ao longo do alimentador',
            'arquivos': []
        },
        'Impacto do regulador (T2F)': {
            'objetivo': 'Avaliar efeito do regulador no T2F',
            'arquivos': []
        },
        'Impacto do regulador (MRT)': {
            'objetivo': 'Avaliar efeito do regulador no MRT',
            'arquivos': []
        },
        'Varredura de Posição': {
            'objetivo': 'Analisar impacto da localização da falta',
            'arquivos': []
        }
    }

    # 1. T2F vs MRT (mesma barra, mesma falta)
    t2f_820_abc = [f for f in categorias['Qualificacao_com_terra']
                   if '820' in f.name and 'abc' in f.name.lower()]
    mrt_820_a = [f for f in categorias['MRT_com_terra']
                 if '820' in f.name and 'falta_a' in f.name.lower()]

    if t2f_820_abc:
        sugestoes['Comparação T2F vs MRT']['arquivos'].append(
            ('T2F_820_ABC', t2f_820_abc[0])
        )
    if mrt_820_a:
        sugestoes['Comparação T2F vs MRT']['arquivos'].append(
            ('MRT_820_A', mrt_820_a[0])
        )

    # 2. Com terra vs sem terra (T2F)
    if categorias['Qualificacao_com_terra'] and categorias['Qualificacao_sem_terra']:
        for barra in ['820', '818', '822']:
            t2f_com = [f for f in categorias['Qualificacao_com_terra']
                       if barra in f.name and 'abc' in f.name.lower() and 'm10p' not in f.name.lower()]
            t2f_sem = [f for f in categorias['Qualificacao_sem_terra']
                       if barra in f.name and 'abc' in f.name.lower() and 'm10p' not in f.name.lower()]

            if t2f_com and t2f_sem:
                sugestoes['Com Terra vs Sem Terra (T2F)']['arquivos'].append(
                    (f'T2F_COM_terra_B{barra}', t2f_com[0])
                )
                sugestoes['Com Terra vs Sem Terra (T2F)']['arquivos'].append(
                    (f'T2F_SEM_terra_B{barra}', t2f_sem[0])
                )
                break

    # 3. Com terra vs sem terra (MRT)
    if categorias['MRT_com_terra'] and categorias['MRT_sem_terra']:
        mrt_com = [f for f in categorias['MRT_com_terra'] if 'falta_a' in f.name.lower()]
        mrt_sem = [f for f in categorias['MRT_sem_terra'] if 'falta_a' in f.name.lower()]

        if mrt_com:
            sugestoes['Com Terra vs Sem Terra (MRT)']['arquivos'].append(
                ('MRT_COM_terra', mrt_com[0])
            )
        if mrt_sem:
            sugestoes['Com Terra vs Sem Terra (MRT)']['arquivos'].append(
                ('MRT_SEM_terra', mrt_sem[0])
            )

    # 4. Tipos de falta (T2F)
    for tipo in ['ABC', 'AB', 'BC', 'AC']:
        cat_key = f'faltas_{tipo}'
        if categorias[cat_key]:
            t2f_files = [f for f in categorias[cat_key] if 'qualificacao' in f.name.lower()]
            if t2f_files:
                sugestoes['Diferentes Tipos de Falta']['arquivos'].append(
                    (f'T2F_Falta_{tipo}', t2f_files[0])
                )

    # 5. Diferentes barras (T2F)
    for barra in ['818', '820', '822']:
        cat_key = f'barras_{barra}'
        if categorias[cat_key]:
            t2f_files = [f for f in categorias[cat_key]
                         if 'qualificacao' in f.name.lower() and 'abc' in f.name.lower() and 'm10p' not in f.name.lower()]
            if t2f_files:
                sugestoes['Comparação de Barras']['arquivos'].append(
                    (f'T2F_Barra_{barra}', t2f_files[0])
                )

    # 6. Com e sem regulador (T2F)
    if categorias['Qualificacao_com_regulador'] and categorias['Qualificacao_sem_regulador']:
        com_reg = [f for f in categorias['Qualificacao_com_regulador']
                   if '820' in f.name and 'abc' in f.name.lower() and 'm10p' not in f.name.lower()]
        sem_reg = [f for f in categorias['Qualificacao_sem_regulador']
                   if '820' in f.name and 'abc' in f.name.lower() and 'm10p' not in f.name.lower()]

        if com_reg:
            sugestoes['Impacto do regulador (T2F)']['arquivos'].append(
                ('T2F_COM_regulador', com_reg[0])
            )
        if sem_reg:
            sugestoes['Impacto do regulador (T2F)']['arquivos'].append(
                ('T2F_SEM_regulador_SR', sem_reg[0])
            )

    # 7. Com e sem regulador (MRT)
    if categorias['MRT_com_regulador'] and categorias['MRT_sem_regulador']:
        mrt_com = categorias['MRT_com_regulador'][:1]
        mrt_sem = categorias['MRT_sem_regulador'][:1]

        if mrt_com:
            sugestoes['Impacto do regulador (MRT)']['arquivos'].append(
                ('MRT_COM_regulador', mrt_com[0])
            )
        if mrt_sem:
            sugestoes['Impacto do regulador (MRT)']['arquivos'].append(
                ('MRT_SEM_regulador_SR', mrt_sem[0])
            )

    # 8. Varredura de posição
    if categorias['varredura_posicao']:
        for arquivo in categorias['varredura_posicao'][:5]:
            match = re.search(r'm10p(\d{2})', arquivo.name.lower())
            if match:
                posicao = match.group(1)
                sugestoes['Varredura de Posição']['arquivos'].append(
                    (f'Posicao_{posicao}%', arquivo)
                )

    # Exibe sugestões
    for analise, info in sugestoes.items():
        if info['arquivos']:
            print(f"\n📌 {analise}")
            print(f" Objetivo: {info['objetivo']}")
            print(f" Arquivos sugeridos:")
            for nome, arquivo in info['arquivos']:
                print(f" ✓ {nome}: {arquivo.name}")

    return sugestoes


In [ ]:
# ======================================================================
# SEÇÃO 11 – EXECUTAR ANÁLISE DE ESTRUTURA E SUGESTÕES
# ======================================================================

PASTA_DADOS = pasta_entrada  # pode usar a mesma pasta_entrada dos HDF5

print("🔍 Analisando estrutura de arquivos...")
print("\n📝 Padrões de identificação:")
print(" • T2F: arquivos com 'Qualificacao'")
print(" • MRT: arquivos com 'MRT'")
print(" • SR = Sem regulador (nomes com '_SR_')")
print(" • sem_terra = Sem aterramento (nomes com '_sem_terra_')")

categorias, todos_arquivos = analisar_estrutura_arquivos(PASTA_DADOS)
exibir_resumo_categorias(categorias)
sugestoes = sugerir_arquivos_tese(categorias)